In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.13 Scattering, Tunneling, and Wave-Packet Dynamics

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.13",
    title="Scattering, Tunneling, and Wave-Packet Dynamics",
    blurb="Motion at last. A particle that is not bound is a wave packet, and we "
    "set it moving by splitting the time-evolution operator into a position step and "
    "a momentum step — a few lines of Fourier transforms that conserve probability "
    "exactly. The packet spreads as it travels, reflects off barriers it has the "
    "energy to cross, and tunnels through barriers it does not — the quantum motions "
    "behind radioactive decay, the tunneling microscope, and the diode.",
    difficulty="advanced",
    estimate="170–210 min",
)

## Notebook overview

Every notebook of Movement II so far has found *stationary* states — the bound levels of a well, the
rungs of the oscillator, things that sit still. This one, the movement's finale, finally introduces
**motion**. A particle that is not bound is a **wave packet** — a normalizable superposition of the
non-normalizable momentum eigenstates of [§6.9](position-representation.ipynb) — and watching it move shows everything the classical
picture forbids. It **spreads** as it travels (unlike the oscillator's coherent state, which a
restoring force holds together); it **reflects** off barriers it has more than enough energy to cross;
and it **tunnels** through barriers it could never surmount classically. That last effect alone
explains how stars burn, how nuclei decay, and how we image single atoms.

The engine is a small, beautiful algorithm: the **split-step Fourier propagator**. The time-evolution
operator $e^{-iHt/\hbar}$ of [§6.7](time-evolution.ipynb) does not factor simply, because the kinetic and potential pieces of
$H=\hat T+\hat V$ do not commute — but the **Strang splitting** $e^{-iH\,dt/\hbar}\approx e^{-i\hat V
dt/2\hbar}e^{-i\hat T dt/\hbar}e^{-i\hat V dt/2\hbar}$ is accurate to $O(dt^3)$ per step, and it works
because $\hat V$ is diagonal in *position* (a multiplication) while $\hat T=\hat p^2/2m$ is diagonal in
*momentum* (a multiplication after a Fourier transform). So each step is a phase, an FFT, a phase, an
inverse FFT, a phase — every factor a pure phase, so the propagator is **unitary by construction** and
conserves probability *exactly*. We derive it, verify its unitarity to a part in $10^{12}$, and turn it
loose.

Then the physics. A **free** packet spreads — we derive the spreading law in-cell rather than quote it,
minding the subtle difference between the wave-function width and the probability-density width. Sent
at a **barrier** it splits into transmitted and reflected parts with $T+R=1$. When the barrier is
*higher* than the particle's energy, classical mechanics forbids transmission entirely, yet the packet
**tunnels** through with a probability that falls *exponentially* with the barrier width — the
mechanism of alpha decay, the **scanning tunneling microscope**, and the tunnel diode. And when the
energy *exceeds* the barrier, classical mechanics predicts certain transmission, yet the packet partly
**reflects** — except at **resonances**, where standing-wave interference inside the barrier makes it
perfectly transparent (the Ramsauer–Townsend effect, the same physics as an anti-reflection coating).

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — the split-step propagator via `numpy.fft.fft`/`ifft`/
`fftfreq`, and `numpy.trapezoid` (or masked sums) for the transmission and reflection probabilities.

> **Conventions and method notes.** $\hbar=m=1$. The grid is $N$ points on a box $[-L/2,L/2]$ with the
> **periodic** boundaries of the FFT, so the box must be large enough that the packet never wraps
> around within the simulation (we use $L=160$, comfortably larger than the packet's travel; absorbing
> boundaries are the alternative for long runs). A Gaussian packet is $\psi\propto e^{-(x-x_0)^2/2\sigma^2}
> e^{ik_0x}$ — note that with this **wave-function width** $\sigma$, the **probability-density** width
> (standard deviation of $|\psi|^2$) is $\sigma/\sqrt2$, a distinction we keep careful track of. See
> Griffiths (scattering, tunneling); Tannor, *A Time-Dependent Perspective* (the split-step method); and
> Notebooks [§6.7](time-evolution.ipynb) (the time-evolution operator), [§6.9](position-representation.ipynb) (momentum eigenstates, the Fourier transform), [§6.11](bound-states-1d.ipynb)
> (the leaking tails, double-well tunneling), [§6.12](harmonic-oscillator.ipynb) (the non-spreading coherent state).

## Theory in brief

### Wave packets and free motion

A physical travelling particle is a normalizable **wave packet**, a superposition of momentum
eigenstates ([§6.9](position-representation.ipynb)),

```{math}
:label: eq-packet
\psi(x,0)\propto e^{-(x-x_0)^2/2\sigma^2}\,e^{ik_0x},\qquad v_{\text{group}}=\frac{\hbar k_0}{m},\qquad |\psi|^2\ \text{has density width}\ \sigma/\sqrt2 .
```

A **free** packet spreads: its different-$k$ components move at different speeds, so it broadens over
time — the contrast to the coherent state of [§6.12](harmonic-oscillator.ipynb), which a restoring potential prevents from spreading. We
derive the density-width spreading law in-cell.

### The split-step Fourier propagator

To evolve $\psi$ under $H=\hat T+\hat V$, use the **Strang splitting**

```{math}
:label: eq-split-step
e^{-iH\,dt/\hbar}\approx e^{-i\hat V dt/2\hbar}\,e^{-i\hat T dt/\hbar}\,e^{-i\hat V dt/2\hbar}\quad(\text{error }O(dt^3)) ,
```

exploiting that $\hat V$ is diagonal in position and $\hat T=\hat p^2/2m$ is diagonal in momentum. Each
step: multiply by $e^{-iV dt/2\hbar}$ (position), FFT, multiply by $e^{-i\hbar k^2 dt/2m}$ (momentum),
inverse FFT, multiply by $e^{-iV dt/2\hbar}$. Every factor is a pure phase, so the propagator is
**unitary** — probability is conserved exactly.

### Barrier scattering: transmission and reflection

A packet of mean energy $E$ sent at a rectangular barrier (height $V_0$, width $a$) splits,

```{math}
:label: eq-scattering
T+R=1 ,
```

into a transmitted part (probability $T$, beyond the barrier) and a reflected part ($R$). The packet
has an energy *spread*, so its $T,R$ match the plane-wave formulas approximately.

### Tunneling

When $V_0>E$ the barrier is **classically forbidden**, yet the quantum packet tunnels with nonzero $T$
that falls exponentially with the width,

```{math}
:label: eq-tunneling
T\sim e^{-2\kappa a},\qquad \kappa=\frac{\sqrt{2m(V_0-E)}}{\hbar} ,
```

the mechanism of **alpha decay**, the **scanning tunneling microscope** (its exponential
distance-sensitivity gives atomic resolution), the tunnel diode, and stellar fusion. The wave function
does not vanish in the barrier — it decays exponentially (the leaking tails of [§6.11](bound-states-1d.ipynb), now traversed).

### Over-barrier reflection and resonances

When $E>V_0$, classical mechanics predicts certain transmission, yet the packet partly **reflects**
($R>0$) — except at **resonances**,

```{math}
:label: eq-scatter-resonance
k_{\text{in}}a=n\pi\ \Longrightarrow\ T=1,\qquad k_{\text{in}}=\frac{\sqrt{2m(E-V_0)}}{\hbar} ,
```

where the barrier width is a whole number of interior half-wavelengths and standing-wave interference
makes it transparent (the **Ramsauer–Townsend** effect; an anti-reflection coating).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation

from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

HBAR = 1.0
MASS = 1.0


def gaussian_packet(x, x0, k0, sigma):
    """A normalized Gaussian wave packet $\\propto e^{-(x-x_0)^2/2\\sigma^2}e^{ik_0x}$ {eq}`eq-packet`.

    Centered at $x_0$ with mean momentum $\\hbar k_0$ (group velocity $\\hbar k_0/m$) and **wave-function**
    width $\\sigma$ — so the probability density $|\\psi|^2$ has width $\\sigma/\\sqrt2$. Normalized with
    ``numpy.trapezoid``.
    """
    psi = np.exp(-((x - x0) ** 2) / (2 * sigma**2)) * np.exp(1j * k0 * x)
    return psi / np.sqrt(np.trapezoid(np.abs(psi) ** 2, x))


def split_step(psi, V, dt, n_steps, x):
    """Propagate $\\psi$ for ``n_steps`` of $dt$ by the split-step Fourier method {eq}`eq-split-step`.

    The Strang step $e^{-iV dt/2\\hbar}\\,$ (position) $\\to$ FFT $\\to e^{-i\\hbar k^2 dt/2m}$ (momentum)
    $\\to$ IFFT $\\to e^{-iV dt/2\\hbar}$, with $k=2\\pi\\,$``numpy.fft.fftfreq``. Every factor is a pure
    phase, so the propagator is **unitary** and conserves the norm exactly; the splitting error is
    $O(dt^3)$ per step. The box must be large enough that the packet does not wrap around the periodic
    FFT boundaries.

    Parameters
    ----------
    psi : numpy.ndarray
        The initial wave function on the grid.
    V : numpy.ndarray
        The (static) potential on the grid.
    dt : float
        The time step.
    n_steps : int
        Number of steps.
    x : numpy.ndarray
        The position grid.

    Returns
    -------
    numpy.ndarray
        The evolved wave function.
    """
    n = len(x)
    dx = x[1] - x[0]
    k = 2 * np.pi * np.fft.fftfreq(n, d=dx)
    half_V = np.exp(-1j * V * dt / (2 * HBAR))  # half potential step
    full_T = np.exp(
        -1j * HBAR * k**2 * dt / (2 * MASS)
    )  # full kinetic step (in momentum space)
    out = psi.copy()
    for _ in range(n_steps):
        out = half_V * out
        out = np.fft.ifft(full_T * np.fft.fft(out))
        out = half_V * out
    return out


def density_width(psi, x):
    """The probability-density width (standard deviation of $|\\psi(x)|^2$), via ``numpy.trapezoid``."""
    density = np.abs(psi) ** 2
    density = density / np.trapezoid(density, x)
    mean = np.trapezoid(x * density, x)
    return np.sqrt(np.trapezoid((x - mean) ** 2 * density, x))


def transmission_reflection(psi, x, barrier_half_width, pad=5.0):
    """Transmission and reflection probabilities of a scattered packet {eq}`eq-scattering`.

    Integrates $|\\psi|^2$ beyond the barrier ($x>a/2+$``pad``, transmitted) and before it ($x<-a/2-$
    ``pad``, reflected) with ``numpy.trapezoid``; the ``pad`` keeps the integration clear of any density
    still inside the barrier region. Returns $(T,R)$, which should satisfy $T+R=1$.
    """
    right = x > barrier_half_width + pad
    left = x < -barrier_half_width - pad
    T = np.trapezoid(np.abs(psi[right]) ** 2, x[right])
    R = np.trapezoid(np.abs(psi[left]) ** 2, x[left])
    return T, R


def plane_wave_T(E, V0, a):
    """The exact plane-wave transmission probability through a rectangular barrier {eq}`eq-tunneling`, {eq}`eq-scatter-resonance`.

    Uses $T=[1+V_0^2\\sinh^2(\\kappa a)/4E(V_0-E)]^{-1}$ for $E<V_0$ (tunneling, $\\kappa=\\sqrt{2m(V_0-E)}/
    \\hbar$) and $T=[1+V_0^2\\sin^2(k_{\\text{in}}a)/4E(E-V_0)]^{-1}$ for $E>V_0$ (with $k_{\\text{in}}=
    \\sqrt{2m(E-V_0)}/\\hbar$), the analytic comparison for the packet results.
    """
    if E < V0:
        kappa = np.sqrt(2 * MASS * (V0 - E)) / HBAR
        return 1.0 / (1 + V0**2 * np.sinh(kappa * a) ** 2 / (4 * E * (V0 - E)))
    k_in = np.sqrt(2 * MASS * (E - V0)) / HBAR
    return 1.0 / (1 + V0**2 * np.sin(k_in * a) ** 2 / (4 * E * (E - V0)))


# the simulation grid (large box so the packet never wraps around the periodic boundaries)
L_BOX, N_GRID = 160.0, 2048
X = np.linspace(-L_BOX / 2, L_BOX / 2, N_GRID)
DT = 0.01

## Exercise 1 — The split-step Fourier propagator

Build the split-step Fourier propagator and verify it conserves probability exactly. The Strang
splitting $e^{-iH\,dt/\hbar}\approx e^{-i\hat V dt/2\hbar}e^{-i\hat T dt/\hbar}e^{-i\hat V dt/
2\hbar}$ alternates a position-space phase and a momentum-space phase, and because every factor is
a pure phase it is unitary {eq}`eq-split-step`.

1. Recall the splitting: $\hat V$ is diagonal in position (multiply by $e^{-iVdt/2\hbar}$) and
   $\hat T=\hat p^2/2m$ is diagonal in momentum (multiply by $e^{-i\hbar k^2dt/2m}$ after
   `numpy.fft.fft`, with $k=2\pi\,$`numpy.fft.fftfreq`).
2. The step is half-$V$ $\to$ FFT $\to$ full-$T$ $\to$ IFFT $\to$ half-$V$ (the `split_step`
   helper).
3. Evolve a free packet for many steps and check the norm $\int|\psi|^2dx$ against its initial
   value (`numpy.trapezoid`).
4. Note the unitarity is automatic — each factor is a phase. Frame: the workhorse of
   time-dependent quantum dynamics.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    norms[-1],
    norms[0],
    "the split-step Fourier propagator is unitary — probability is conserved over thousands of steps",
    rtol=1e-5,
)

## Exercise 2 — Free-packet spreading

Evolve a *free* Gaussian packet and show it **spreads**, and verify the spreading law by deriving
it in-cell from the momentum distribution rather than quoting it. The density width grows from
$\sigma/\sqrt2$ as $w(t)=(\sigma/\sqrt2)\sqrt{1+(\hbar t/m\sigma^2)^2}$ {eq}`eq-packet`.

1. Build a stationary Gaussian ($k_0=0$) of wave-function width $\sigma$; note its **density**
   width starts at $\sigma/\sqrt2$ (since $|\psi|^2\propto e^{-x^2/\sigma^2}$ has standard
   deviation $\sigma/\sqrt2$ — the wave-function-vs-density distinction).
2. Evolve it freely and measure the density width with the `density_width` helper at several
   times.
3. Derive the law in-cell: the momentum amplitude is $\varphi(k)\propto e^{-\sigma^2k^2/2}$
   (Fourier reciprocity, [§6.9](position-representation.ipynb)), and free evolution attaches $e^{-i\hbar k^2t/2m}$ to each
   component; carrying this through gives $w(t)=(\sigma/ \sqrt2)\sqrt{1+(\hbar t/m\sigma^2)^2}$.
4. Confirm the simulated widths match it and grow monotonically. Frame: unlike the coherent state
   ([§6.12](harmonic-oscillator.ipynb)), a free packet has no restoring force and spreads.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    np.all(np.diff(widths_sim) > 0) and np.allclose(widths_sim, widths_law, rtol=2e-2),
    "a free wave packet spreads: the density width grows monotonically and matches the in-cell-derived law w(t)=(σ/√2)√(1+(ℏt/mσ²)²)",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Barrier scattering: $T+R=1$

Send a packet at a barrier and measure the transmitted and reflected probabilities, confirming
that probability splits but is conserved, $T+R=1$ {eq}`eq-scattering`.

1. Build a rectangular barrier (height $V_0$, width $a$) and a packet of mean momentum $k_0$,
   hence mean energy $E=\hbar^2k_0^2/2m$.
2. Evolve with `split_step` until the packet has fully scattered (the transmitted and reflected
   parts are clear of the barrier).
3. Integrate $|\psi|^2$ beyond the barrier ($T$) and before it ($R$) with the
   `transmission_reflection` helper.
4. Confirm $T+R=1$ to high precision — probability is conserved (the propagator is unitary).
   Frame: probability splits and is conserved.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    T + R,
    1.0,
    "scattering conserves probability: T + R = 1 (the split-step propagator is unitary)",
    rtol=2e-3,
)

## Exercise 4 — Tunneling through a forbidden barrier

Send a packet at a barrier *higher* than its energy ($V_0>E$) — classically forbidden — and show
it **tunnels** through with nonzero $T$, the exponentially-decaying wave function inside the
barrier never quite reaching zero {eq}`eq-tunneling`.

1. Set $V_0>E$ (the barrier is classically impenetrable).
2. Evolve with `split_step` and measure $T$ with `transmission_reflection`.
3. Confirm $T>0$ — the packet tunnels — and compare to the analytic plane-wave $T$
   (`plane_wave_T`, approximate because the packet has an energy spread), with $T+R=1$.
4. Note the wave function inside the barrier is not zero but an exponential decay $e^{-\kappa x}$
   (the leaking tails of [§6.11](bound-states-1d.ipynb), now traversed). Frame: a quantum effect with no classical analogue.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    T_tun > 0.001
    and np.isclose(T_tun + R_tun, 1.0, atol=2e-3)
    and abs(T_tun - T_analytic) < 0.05,
    "a packet tunnels through a classically forbidden barrier (T>0 for V0>E), with T+R=1 and T near the analytic plane-wave value",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — Exponential suppression of tunneling

Show that the transmission falls **exponentially** with the barrier width: $\ln T$ is linear in
$a$ with slope $-2\kappa$, $\kappa=\sqrt{2m(V_0-E)}/\hbar$ {eq}`eq-tunneling`.

1. Compute the analytic tunneling probability $T$ (the `plane_wave_T` helper) for a series of
   barrier widths $a$.
2. Plot $\ln T$ against $a$.
3. Confirm the large-$a$ slope is $-2\kappa$ (a `numpy.polyfit` of $\ln T$ vs $a$ in the
   thick-barrier regime).
4. Connect to alpha decay and the STM — a small change in width changes $T$, and hence a tunneling
   current, by orders of magnitude. Frame: tunneling is exquisitely sensitive to the barrier.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    slope,
    -2 * kappa,
    "tunneling is exponentially suppressed with barrier width: ln T is linear in a with slope −2κ",
    rtol=5e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — Over-barrier reflection and resonant transmission

Send a packet at a barrier *lower* than its energy ($E>V_0$) and find that it still partly
**reflects** ($R>0$, classically forbidden), except at **resonances** where the barrier width is a
whole number of interior half-wavelengths ($k_{\text{in}}a=n\pi$) and transmission is perfect
{eq}`eq-scatter-resonance`.

1. For $E>V_0$, compute the analytic transmission $T(E)$ (the `plane_wave_T` helper) across a
   range of energies for a fixed barrier.
2. Observe $T<1$ in general — so $R=1-T>0$, a reflection classical mechanics forbids, arising from
   the abrupt potential edges.
3. Find the resonance condition $k_{\text{in}}a=n\pi$ with $k_{\text{in}}=\sqrt{2m(E-V_0)}/\hbar$,
   and confirm $T=1$ there.
4. Verify perfect transmission at a resonant energy. Frame: a purely quantum reflection, and a
   standing-wave interference that makes the barrier transparent (Ramsauer–Townsend; the
   anti-reflection-coating analogy).

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    plane_wave_T(E_nonres, V0_res, a_res) < 0.99
    and np.allclose(T_at_resonances, 1.0, atol=1e-6),
    "over-barrier reflection occurs (R>0 for E>V0) except at resonances k_in·a=nπ, where transmission is perfect (T=1)",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — Tunneling and the scanning tunneling microscope *(student)*

Estimate how the tunneling current in a scanning tunneling microscope (STM) depends on the
tip–surface distance, and explain why the exponential law gives the instrument its atomic
resolution {eq}`eq-tunneling`.

1. Model the vacuum gap between tip and surface as a barrier of height the **work function**
   $W\approx4.5\,$eV.
2. The decay constant is $\kappa=\sqrt{2m_eW}/\hbar\approx0.51\sqrt{W/ \mathrm{eV}}\
   \text{Å}^{-1}$; the tunneling current scales as the transmission $I\propto e^{-2\kappa d}$ with
   the gap $d$.
3. Compute the ratio of currents for gaps differing by $1\,$Å and show it is roughly an order of
   magnitude.
4. Explain: because the current changes about tenfold per ångström, a sub-ångström height change
   is easily read off the current — so the STM maps a surface atom by atom. Frame: the tunneling
   of this notebook is the working principle of a real instrument.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    5 < ratio_per_angstrom < 20,
    "the STM's atomic resolution comes from the exponential distance-dependence of tunneling: the current changes ~an order of magnitude per ångström of gap",
)

## Exercise 8 — Quantum motion: spreading, tunneling, and the failure of the classical picture

A particle that is not bound is a wave packet, and watching it move — by splitting the evolution
operator into a position kick and a momentum drift, a phase and a Fourier transform and a phase — shows
everything the classical picture forbids. It **spreads** as it travels, because it is a superposition
of momenta that disperse. It **reflects** off barriers it has the energy to cross, because of the sharp
edges. And it **tunnels** through barriers it could never surmount, because the wave function is never
quite zero in the forbidden region. Tunnelling alone explains how stars burn, how nuclei decay, and how
we image single atoms.

**This exercise (synthesis).** No new computation: the catalogue of quantum motions is the result. The
split-step method is almost embarrassingly simple — a phase, a Fourier transform, a phase — yet it
shows a particle tunnelling through a wall in real time. With this, **Movement II is complete**: we
took the formalism to the continuum ([§6.9](position-representation.ipynb)), solved the bound states by diagonalization ([§6.10](schrodinger-on-a-computer.ipynb), [§6.11](bound-states-1d.ipynb)),
met the universal oscillator three ways ([§6.12](harmonic-oscillator.ipynb)), and set wave packets moving through barriers (§6.13) —
the two systems, the oscillator and the barrier, that physics returns to most. **Movement III** now
adds the dimension we have been missing: angular momentum, built from the very same ladder algebra of
[§6.12](harmonic-oscillator.ipynb), assembles the spherical world and, at its summit, the hydrogen atom.

## Notebook summary

Quantum motion through the continuous spectrum — the close of Movement II.

- **Wave packets** {eq}`eq-packet`: a free particle is a normalizable superposition of momentum
  eigenstates ([§6.9](position-representation.ipynb)), with group velocity $\hbar k_0/m$; its density width is $\sigma/\sqrt2$.
- **The split-step propagator** {eq}`eq-split-step`: the Strang splitting (`numpy.fft.fft`/`ifft`),
  half-$V$/full-$T$/half-$V$ — unitary by construction, norm conserved to $\sim10^{-12}$.
- **Spreading** {eq}`eq-packet`: a free packet broadens as $w(t)=(\sigma/\sqrt2)\sqrt{1+(\hbar t/m
  \sigma^2)^2}$ (derived in-cell) — unlike the coherent state ([§6.12](harmonic-oscillator.ipynb)).
- **Scattering and tunnelling** {eq}`eq-scattering`, {eq}`eq-tunneling`: $T+R=1$; for $V_0>E$ the packet
  tunnels with $T\sim e^{-2\kappa a}$ — alpha decay, the STM, the tunnel diode.
- **Over-barrier resonances** {eq}`eq-scatter-resonance`: for $E>V_0$, $R>0$ except at $k_{\text{in}}a=n\pi$
  where $T=1$ (Ramsauer–Townsend, an anti-reflection coating).

A phase, a Fourier transform, a phase — and a particle tunnels through a wall in real time. Movement
II is complete; Movement III adds the missing dimension.

## Outlook

- **Three-dimensional scattering**: cross sections, partial waves, the Born approximation (a horizon;
  time-dependent transitions are [§6.24](time-dependent-perturbation.ipynb)).
- **The angular-momentum algebra and three dimensions (Movement III, [§6.14](angular-momentum-algebra.ipynb))**: the same ladder method of
  [§6.12](harmonic-oscillator.ipynb), now for rotations — building toward the hydrogen atom.
- **Tunnelling in the wild**: alpha decay (Gamow), the STM, Josephson junctions, stellar fusion
  (horizons; the STM ties to surface science).
- **Absorbing boundary conditions** for larger time-dependent simulations (a computational horizon).
- **Cross-reference** [§6.7](time-evolution.ipynb) (time evolution), [§6.9](position-representation.ipynb) (momentum eigenstates, the Fourier transform), [§6.11](bound-states-1d.ipynb)
  (leaking tails, double-well tunnelling), [§6.12](harmonic-oscillator.ipynb) (the non-spreading coherent state), and forward to
  [§6.14](angular-momentum-algebra.ipynb), [§6.24](time-dependent-perturbation.ipynb).

In [ ]:
from ecp.style import footer

footer()